# NB-Ramen — Pre-full CUDA smoke trên Kaggle

Trong Settings bật **Internet** và chọn **GPU** rồi chạy các cell theo thứ tự.
Notebook tải đúng commit `26a7cd7c847b6630841dbae58067e5bb124f2f9d`, tạo Python 3.11
riêng với PyTorch 2.4.1/cu121 và CLIP commit cố định. Không cần restart kernel.

Chạy **14 smoke**, mỗi run 256 mẫu, gồm bảy phương pháp chính và các control.
Đây là evidence **noncanonical**. Full matrix 252 runs chỉ được dry-plan, không chạy.
Batch size 100 giữ nguyên. Code dùng một GPU; hai GPU không tự cộng VRAM.
Nếu thiếu VRAM, notebook ghi lỗi và dừng, không tự giảm batch hoặc precision.

Có thể điền đường dẫn archive chính thức `CIFAR-100-C.tar` trong Kaggle Input
ở cell cấu hình; để trống để tải từ Zenodo. Archive phải khớp checksum chính thức.
Data và môi trường nằm trong `/tmp`; evidence ở `/kaggle/working/nb-ramen-evidence-<revision>`.
Nếu phiên dừng giữa một run, thư mục dở dang được giữ lại và từ chối resume.
Đổi tên thư mục run lỗi để lưu chẩn đoán rồi chạy lại cell smoke; không sửa JSON.
Sau khi khởi động phiên Kaggle mới cần chạy lại setup/data. Resume trong cùng
phiên sẽ skip những run đã được strict validator chấp nhận.

Nguồn thiết lập: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks),
[PyTorch 2.4.1](https://docs.pytorch.org/get-started/previous-versions/),
[uv environments](https://docs.astral.sh/uv/pip/environments/).


In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path

REVISION = "26a7cd7c847b6630841dbae58067e5bb124f2f9d"
WORK = Path("/kaggle/working")
REPO = WORK / "NB-Ramen"
PYTHON = Path("/tmp/nb-ramen-venv/bin/python")
DATA = Path("/tmp/nb-ramen-data")
EVIDENCE = WORK / f"nb-ramen-evidence-{REVISION[:7]}"
RUNTIME = EVIDENCE / "runtime"
# Tùy chọn: ví dụ "/kaggle/input/my-official-cifar100c/CIFAR-100-C.tar"
ARCHIVE_INPUT = ""
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME.mkdir(parents=True, exist_ok=True)
env = dict(os.environ, PYTHONPATH=str(REPO / "src"),
           RAMEN_REPOSITORY=str(REPO), RAMEN_REVISION=REVISION,
           RAMEN_DATA_ROOT=str(DATA), RAMEN_EVIDENCE_ROOT=str(EVIDENCE),
           RAMEN_ARCHIVE_INPUT=ARCHIVE_INPUT, CUDA_VISIBLE_DEVICES="0",
           PYTHONUNBUFFERED="1", UV_CACHE_DIR="/tmp/nb-ramen-uv-cache")

def run(argv, *, log=None, cwd=None):
    argv = [str(x) for x in argv]
    if log is None:
        return subprocess.run(argv, cwd=cwd, env=env, check=True)
    with Path(log).open("w") as handle:
        proc = subprocess.Popen(argv, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
        code = proc.wait()
    if code:
        raise subprocess.CalledProcessError(code, argv)

run(["nvidia-smi"], log=RUNTIME / "nvidia-smi.txt")
if not REPO.exists():
    run(["git", "clone", "--branch", "open-world-gradient-memory", "--single-branch",
         "https://github.com/nguyetbinh/NB-Ramen.git", REPO])
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
if dirty:
    raise RuntimeError("Checkout có thay đổi chưa commit; giữ lại các thay đổi trước khi cập nhật.")
if actual != REVISION:
    run(["git", "fetch", "origin", "open-world-gradient-memory"], cwd=REPO)
    run(["git", "checkout", "--detach", REVISION], cwd=REPO)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == REVISION
(RUNTIME / "git-head.txt").write_text(actual + "\n")
(RUNTIME / "git-status.txt").write_text(dirty)

run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
if not PYTHON.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", PYTHON.parent.parent])
run([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON, "pip==24.2"])
run([PYTHON, "-m", "pip", "install", "--no-cache-dir", "torch==2.4.1", "torchvision==0.19.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"])
run([PYTHON, "-m", "pip", "install", "--no-cache-dir", "numpy==1.26.4", "pillow==10.4.0",
     "pyyaml==6.0.2", "tqdm==4.66.5",
     "git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"])
run([PYTHON, "-m", "pip", "check"], log=RUNTIME / "pip-check.txt")
run([PYTHON, "-m", "pip", "freeze"], log=RUNTIME / "pip-freeze.txt")


## Kiểm tra runtime và tests
Cell phải hoàn tất thành công trước khi tải data/chạy smoke.


In [ ]:
run([PYTHON, '-c', "\nimport json, os, platform, sys\nfrom pathlib import Path\nimport torch, torchvision\nfrom importlib.metadata import distribution, version\nfrom evaluation.evidence import TRACE_SCHEMA_VERSION, SUMMARY_SCHEMA_VERSION\nassert sys.version_info[:2] == (3, 11)\nassert torch.__version__.split('+')[0] == '2.4.1'\nassert torchvision.__version__.split('+')[0] == '0.19.1'\nassert torch.version.cuda == '12.1' and torch.cuda.is_available(), 'CUDA 12.1 runtime unavailable'\nassert (TRACE_SCHEMA_VERSION, SUMMARY_SCHEMA_VERSION) == (3, 4)\nimport subprocess\nfrom runtime.experiment_matrix import build_experiment_matrix, build_command\nprobe_run = build_experiment_matrix(datasets=('CIFAR100C',), streams=('block',),\n                                   methods=('NoAdapt',), seeds=(0,), device='cuda')[0]\nchild_python = build_command(probe_run)[0]\nassert child_python == sys.executable, 'Generated command changed the virtualenv interpreter'\nsubprocess.run([child_python, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__)'], check=True)\nclip_source = json.loads(distribution('clip').read_text('direct_url.json'))\nassert clip_source['vcs_info']['commit_id'] == 'd05afc436d78f1c48dc0dbf8e5980a9d471f35f6'\nfor package, expected in [('numpy','1.26.4'),('pillow','10.4.0'),('pyyaml','6.0.2'),('tqdm','4.66.5')]:\n    assert version(package) == expected, package\nidentity = dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__,\n                cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), platform=platform.platform(),\n                total_vram_gib=torch.cuda.get_device_properties(0).total_memory / 2**30,\n                visible_devices=os.environ.get('CUDA_VISIBLE_DEVICES'), clip_source=clip_source)\n(Path(os.environ['RAMEN_EVIDENCE_ROOT'])/'runtime/device.json').write_text(json.dumps(identity,indent=2))\nprint(json.dumps(identity, indent=2))\n"], cwd=REPO, log=RUNTIME / 'runtime-check.log')

focused = ["tests.test_by_sample_normalization", "tests.test_ramen_cuda_half", "tests.test_entropy_gated_ramen", "tests.test_consensus_ramen",
           "tests.test_oracle_id_gradient_ramen", "tests.test_oracle_consensus_ramen",
           "tests.test_open_set", "tests.test_open_set_metrics", "tests.test_open_set_consensus_analysis",
           "tests.test_ordered_stream_evidence", "tests.test_experiment_matrix"]
run([PYTHON, "-m", "unittest", *focused], cwd=REPO, log=RUNTIME / "focused-tests.log")
run([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py"],
    cwd=REPO, log=RUNTIME / "full-tests.log")
(RUNTIME / "test-status.json").write_text(json.dumps({"focused_exit": 0, "full_exit": 0}))


## Data chính thức và model
Tải archive khoảng 2.92 GB nếu chưa gắn Kaggle Input. Xác minh MD5 archive,
giải nén, tạo inventory SHA-256, xác minh checkpoint, rồi chạy deep preflight.
Không dùng CIFAR-100 sạch hoặc bộ corruption tự tạo thay cho CIFAR-100-C.


In [ ]:
prepare_script = RUNTIME / 'prepare-data.py'
prepare_script.write_text('"""Prepare the checksum-verified official CIFAR-100-C archive and CLIP model."""\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport subprocess\nimport tarfile\n\nfrom runtime.artifact_provenance import (\n    CIFAR100C_OFFICIAL_ACQUISITION, verify_official_cifar100c_archive,\n    generate_cifar100c_provenance, verify_cifar100c_provenance,\n    resolve_clip_model, verify_cached_clip_checkpoint,\n)\nfrom evaluation.evidence import atomic_write_json\n\n\ndef download(url, path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        partial = path.with_suffix(path.suffix + ".part")\n        subprocess.run(["curl", "--fail", "--location", "--retry", "3", "--retry-delay", "5",\n                        "--output", str(partial), url], check=True)\n        partial.replace(path)\n\n\ndef prepare():\n    data = Path(os.environ["RAMEN_DATA_ROOT"])\n    runtime = Path(os.environ["RAMEN_EVIDENCE_ROOT"]) / "runtime"\n    runtime.mkdir(parents=True, exist_ok=True)\n    archive_input = os.environ.get("RAMEN_ARCHIVE_INPUT", "").strip()\n    archive = Path(archive_input) if archive_input else data.parent / "CIFAR-100-C.tar"\n    if archive_input and not archive.is_file():\n        raise FileNotFoundError(f"Attached archive not found: {archive}")\n    if not archive_input:\n        download(CIFAR100C_OFFICIAL_ACQUISITION["url"], archive)\n    print("Verifying the official archive MD5 and size...", flush=True)\n    acquisition = verify_official_cifar100c_archive(archive)\n    atomic_write_json(runtime / "archive-acquisition.json", acquisition)\n    dataset = data / "corruption/CIFAR-100-C"\n    if not dataset.exists():\n        staging = data / "extract-staging"\n        staging.mkdir(parents=True, exist_ok=False)\n        # Permit only the expected dataset tree and regular files/directories.\n        with tarfile.open(archive) as source:\n            members = source.getmembers()\n            for member in members:\n                parts = PurePosixPath(member.name).parts\n                if (not parts or parts[0] != "CIFAR-100-C" or ".." in parts\n                        or not (member.isfile() or member.isdir())):\n                    raise RuntimeError(f"Unexpected archive entry: {member.name}")\n            source.extractall(staging, members=members, filter="data")\n        staged = staging / "CIFAR-100-C"\n        # Inventory is created only for bytes extracted from the verified archive.\n        generate_cifar100c_provenance(staged, acquisition=acquisition)\n        dataset.parent.mkdir(parents=True, exist_ok=True)\n        staged.rename(dataset)\n        staging.rmdir()\n    # On resume, never bless an arbitrary existing tree by rebuilding its sidecar.\n    dataset_provenance = verify_cifar100c_provenance(dataset, exact=True)\n    resolved = resolve_clip_model("clip_vitbase16")\n    cache = Path.home() / ".cache/clip"\n    download(resolved["url"], cache / resolved["filename"])\n    model_provenance = verify_cached_clip_checkpoint("clip_vitbase16", cache)\n    atomic_write_json(runtime / "artifact-provenance.json", {\n        "dataset": dataset_provenance, "model": model_provenance,\n    })\n    print("Official data and model verified.", flush=True)\n\n\nif __name__ == "__main__":\n    prepare()\n')

run([PYTHON, prepare_script], cwd=REPO, log=RUNTIME / "prepare-data.log")
run([PYTHON, "-m", "runtime.preflight", "--data-root", DATA, "--dataset", "CIFAR100C",
     "--deep", "--json"], cwd=REPO, log=RUNTIME / "cifar100c-deep-preflight.json")


## Chạy 14 smoke và strict validation
Giữ batch=100, block=64, source budget=400/domain, prefix=256, CUDA, fast provenance.
Chạy tuần tự trên GPU 0; mỗi method chạy trong process riêng để giải phóng VRAM.
NoAdapt B=1 được tạo riêng cho Ramen B=1. V2 có split byte lock.
Mỗi run có log riêng trong `runtime/prefull-*.log`.

Notebook chỉ đánh dấu artifact smoke hợp lệ sau strict validation. Kết quả
causal sensitivity vẫn cần đọc/đánh giá; không tự công nhận toàn bộ pre-full gate.


In [ ]:
smoke_script = RUNTIME / 'run-smokes.py'
smoke_script.write_text('"""Run the fixed pre-full CUDA smoke controls; never launch the full matrix."""\nimport argparse\nfrom dataclasses import replace\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nfrom runtime.experiment_matrix import (\n    build_canonical_open_set_evidence_matrix, build_open_set_evidence_matrix,\n    build_experiment_matrix, build_command, validate_completed_run,\n)\nfrom runtime.open_set_split_robustness_matrix import SPLIT_SHA256\nfrom evaluation.evidence import atomic_write_json, SUMMARY_SCHEMA_VERSION, TRACE_SCHEMA_VERSION\n\n\ndef plan_smokes(repo, data, evidence):\n    canonical = build_canonical_open_set_evidence_matrix(\n        data_root=data, evidence_dir=evidence / "canonical-not-executed",\n        device="cuda", artifact_provenance="fast",\n    )\n    common = dict(\n        streams=("block",), seeds=(0,), evidence_dir=evidence / "smoke",\n        device="cuda", max_eval_samples=256, artifact_provenance="fast", data_root=data,\n        config_dir=repo / "cfg",\n    )\n    primary = build_open_set_evidence_matrix(ood_ratios=(.3,), **common)\n    ids = ["prefull-noadapt", "prefull-ramen", "prefull-entropy", "prefull-oracle-drop",\n           "prefull-oracle-id", "prefull-consensus", "prefull-oracle-consensus"]\n    runs = [replace(\n        run, run_id=run_id, require_config_lock=run.method != "NoAdapt",\n        reference_trace=None if run.method == "NoAdapt" else common["evidence_dir"] / "prefull-noadapt/trace.jsonl",\n    ) for run, run_id in zip(primary, ids)]\n    b1 = replace(runs[0], run_id="prefull-noadapt-b1", batch_size=1)\n    runs += [b1, replace(runs[1], run_id="prefull-ramen-b1", batch_size=1,\n                         reference_trace=b1.run_dir / "trace.jsonl")]\n    causal = build_experiment_matrix(datasets=("CIFAR100C",), methods=("CausalRamen",), **common)[1]\n    runs.append(replace(\n        causal, run_id="prefull-causal-b100", open_set=True,\n        known_class_split=primary[0].known_class_split, ood_ratio=.3,\n        open_set_per_domain_source_budget=400, require_config_lock=True,\n        reference_trace=runs[0].run_dir / "trace.jsonl",\n    ))\n    split = "open-set-cifar100-name-rank-v2"\n    split_path = (repo / "cfg/research/open-set-cifar100-split-v2.json").resolve()\n    if hashlib.sha256(split_path.read_bytes()).hexdigest() != SPLIT_SHA256[split]:\n        raise RuntimeError("Frozen v2 split bytes changed")\n    v2 = replace(runs[0], run_id="prefull-v2-noadapt", known_class_split=split,\n                 known_class_split_path=split_path, known_class_split_sha256=SPLIT_SHA256[split])\n    runs += [v2, replace(runs[2], run_id="prefull-v2-entropy", known_class_split=split,\n                         known_class_split_path=split_path, known_class_split_sha256=SPLIT_SHA256[split],\n                         reference_trace=v2.run_dir / "trace.jsonl")]\n    zero = replace(runs[0], run_id="prefull-ood0-noadapt", ood_ratio=0.0)\n    runs += [zero, replace(runs[4], run_id="prefull-ood0-oracle-id", ood_ratio=0.0,\n                           reference_trace=zero.run_dir / "trace.jsonl")]\n    if len(runs) != 14 or len({r.run_id for r in runs}) != 14 or len(canonical) != 252:\n        raise RuntimeError("Unexpected plan size")\n    return runs, canonical\n\n\ndef trace(run):\n    return [json.loads(line) for line in (run.run_dir / "trace.jsonl").read_text().splitlines() if line.strip()]\n\n\ndef compare(left, right):\n    if len(left) != len(right) or not left:\n        raise RuntimeError("Sensitivity traces have different lengths or are empty")\n    for a, b in zip(left, right):\n        for field in ("timestep", "sample_idx", "ground_truth_domain", "original_label", "is_ood"):\n            if a[field] != b[field]:\n                raise RuntimeError(f"Sensitivity traces disagree on {field}")\n    def id_accuracy(rows):\n        ids = [r for r in rows if not r["is_ood"]]\n        return sum(r["prediction"] == r["known_label_or_minus_one"] for r in ids) / len(ids) if ids else None\n    return {\n        "samples": len(left),\n        "prediction_disagreement_count": sum(a["prediction"] != b["prediction"] for a, b in zip(left, right)),\n        "pre_prediction_disagreement_count": sum(a["pre_adaptation_prediction"] != b["pre_adaptation_prediction"] for a, b in zip(left, right)),\n        "left_id_accuracy": id_accuracy(left), "right_id_accuracy": id_accuracy(right),\n        "max_absolute_post_ood_score_difference": max(abs(a["post_adaptation_ood_score"] - b["post_adaptation_ood_score"]) for a, b in zip(left, right)),\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--plan-only", action="store_true")\n    args = parser.parse_args()\n    repo = Path(os.environ["RAMEN_REPOSITORY"]).resolve()\n    data = Path(os.environ["RAMEN_DATA_ROOT"]).resolve()\n    evidence = Path(os.environ["RAMEN_EVIDENCE_ROOT"]).resolve()\n    runtime = evidence / "runtime"\n    runtime.mkdir(parents=True, exist_ok=True)\n    runs, canonical = plan_smokes(repo, data, evidence)\n    for name, planned in (("smoke-plan", runs), ("canonical-plan", canonical)):\n        atomic_write_json(runtime / f"{name}.json", {\n            "execution_requested": name == "smoke-plan" and not args.plan_only,\n            "runs": [r.to_dict() for r in planned],\n            "commands": [build_command(r) for r in planned],\n        })\n    print("Plan: 14 smoke runs; 252 canonical runs (plan only).", flush=True)\n    if args.plan_only:\n        return\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA unavailable")\n    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()\n    if head != os.environ["RAMEN_REVISION"] or subprocess.check_output(["git", "status", "--porcelain"], cwd=repo):\n        raise RuntimeError("The experiment checkout must match the pinned revision and be clean")\n    status = {"classification": "noncanonical_pilot", "trace_schema": TRACE_SCHEMA_VERSION,\n              "summary_schema": SUMMARY_SCHEMA_VERSION, "completed": [], "status": "running"}\n    atomic_write_json(runtime / "smoke-status.json", status)\n    completed = {}\n    for index, run in enumerate(runs, 1):\n        print(f"[{index}/{len(runs)}] {run.run_id}", flush=True)\n        try:\n            if not run.run_dir.exists():\n                with (runtime / f"{run.run_id}.log").open("w") as log:\n                    result = subprocess.run(build_command(run), cwd=repo, stdout=log, stderr=subprocess.STDOUT)\n                if result.returncode:\n                    tail = (runtime / f"{run.run_id}.log").read_text(errors="replace")[-8000:]\n                    print(tail, flush=True)\n                    raise RuntimeError(f"{run.run_id} failed with exit {result.returncode}")\n            # A partial or corrupted directory is rejected. Do not silently overwrite it.\n            checked = validate_completed_run(run)\n            if checked["manifest"]["git"].get("commit") != head or checked["manifest"]["git"].get("dirty"):\n                raise RuntimeError("Completed run was produced by a different or dirty revision")\n            if run.reference_trace is not None:\n                baseline = completed[run.reference_trace]\n                if checked["summary"]["stream_fingerprint"] != baseline["summary"]["stream_fingerprint"]:\n                    raise RuntimeError("Paired stream fingerprints disagree")\n            completed[run.run_dir / "trace.jsonl"] = checked\n            status["completed"].append({"run_id": run.run_id,\n                                        "fingerprint": checked["summary"]["stream_fingerprint"],\n                                        "id_accuracy": checked["summary"]["open_set"]["id_accuracy"]})\n            atomic_write_json(runtime / "smoke-status.json", status)\n        except Exception as exc:\n            status.update(status="failed", failed_run=run.run_id, error=str(exc))\n            atomic_write_json(runtime / "smoke-status.json", status)\n            raise\n    by_id = {r.run_id: r for r in runs}\n    try:\n        causal_ids = ["prefull-ramen", "prefull-ramen-b1", "prefull-causal-b100"]\n        if len({completed[by_id[i].run_dir / "trace.jsonl"]["summary"]["stream_fingerprint"] for i in causal_ids}) != 1:\n            raise RuntimeError("Causal controls must share the exact stream fingerprint")\n        sensitivity = {\n            "status": "measured_review_required",\n            "pretrained_ramen_b100_vs_noadapt_b100": compare(trace(by_id["prefull-ramen"]), trace(by_id["prefull-noadapt"])),\n            "pretrained_ramen_b1_vs_noadapt_b1": compare(trace(by_id["prefull-ramen-b1"]), trace(by_id["prefull-noadapt-b1"])),\n            "packaging_ramen_b100_vs_b1": compare(trace(by_id[causal_ids[0]]), trace(by_id[causal_ids[1]])),\n            "causal_ramen_b1_vs_causal_b100": compare(trace(by_id[causal_ids[1]]), trace(by_id[causal_ids[2]])),\n        }\n        atomic_write_json(runtime / "causal-sensitivity.json", sensitivity)\n        null_rows = trace(by_id["prefull-ood0-oracle-id"])\n        for row in null_rows:\n            if row["retrieved_ood_fraction"] != 0 or row["retrieved_ood_weight_fraction"] != 0:\n                raise RuntimeError("OOD=0 control retrieved OOD supports")\n            cosine = row["ramen_vs_oracle_id_cosine"]\n            disagreement = row["ramen_vs_oracle_id_sign_disagreement"]\n            if cosine is not None and abs(cosine - 1) > 1e-5:\n                raise RuntimeError("OOD=0 defined cosine differs from one beyond 1e-5")\n            if disagreement is not None and disagreement != 0:\n                raise RuntimeError("OOD=0 sign disagreement is nonzero")\n        status.update(status="smokes_validated_causal_review_required", null_control="passed",\n                      full_matrix_executed=False)\n    except Exception as exc:\n        status.update(status="control_check_failed", error=str(exc))\n        raise\n    finally:\n        atomic_write_json(runtime / "smoke-status.json", status)\n    print(json.dumps(status, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')

# Log và script được giữ ngoài checkout để Git identity của thí nghiệm sạch.
try:
    run([PYTHON, smoke_script], cwd=REPO, log=RUNTIME / "smoke-execution.log")
finally:
    archive_path = shutil.make_archive(str(EVIDENCE), "zip",
                                       root_dir=WORK, base_dir=EVIDENCE.name)
    print("Evidence archive:", archive_path)


## Xem kết quả và tải evidence
Cell này có thể chạy riêng sau lỗi để đóng gói cả log chẩn đoán. Tải ZIP về
trước khi kết thúc phiên; dùng **Save Version** để lưu notebook/output.
Nếu GPU hết VRAM, giữ log và chuyển sang runner đủ VRAM cho batch=100.
Không giảm batch trong cùng run ID rồi gọi kết quả đó là cùng smoke.

File cần đọc: `runtime/smoke-status.json`, `runtime/causal-sensitivity.json`,
`runtime/device.json` và các `summary.json`. Gửi ZIP để kiểm tra trước full run.


In [ ]:
for name in ("smoke-status.json", "causal-sensitivity.json"):
    p = RUNTIME / name
    if p.exists():
        print(name, p.read_text())
archive_path = shutil.make_archive(str(EVIDENCE), "zip",
                                   root_dir=WORK, base_dir=EVIDENCE.name)
from IPython.display import display, FileLink
display(FileLink(archive_path))
